# GLITCH_OPUS — X_GLITCH Fine-Tuning (Google Colab)
Glasses Software | Siber Operasyon Baskani

Base: Qwen 3.5 9B | Method: QLoRA (Unsloth) | Dataset: 500 entry

In [ ]:
# 1. Unsloth + bagimliliklari yukle
!pip install unsloth torch transformers datasets trl accelerate bitsandbytes
!pip install xformers --index-url https://download.pytorch.org/whl/cu124

In [ ]:
# 2. GitHub'dan repo'yu cek (veya Google Drive'a yukle)
!git clone https://github.com/glassesglitchstudio/glasses-cat-ai.git
%cd glasses-cat-ai

# YA DA: Google Drive'dan yukle
# from google.colab import drive
# drive.mount('/content/drive')
# %cp /content/drive/MyDrive/glitch_dataset.json .

In [ ]:
# 3. GPU kontrol
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# 4. Model yukle (QLoRA 4-bit NF4)
from unsloth import FastLanguageModel
import torch

BASE_MODEL = "unsloth/qwen2.5-coder-14b"  # veya "unsloth/qwen3.5-9b"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 32
LORA_ALPHA = 64

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map="auto",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                    "embed_tokens", "lm_head"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=MAX_SEQ_LENGTH,
    use_rslora=True,
)
print("Model + LoRA adapter yuklendi")

In [ ]:
# 5. Dataset yukle
import json
from datasets import load_dataset

dataset = load_dataset("json", data_files="glitch_dataset.json", split="train")
print(f"{len(dataset)} egitim ornegi yuklendi")

# Identity formatlama
GLASSES_IDENTITY = """Sen Glasses Software Yazilim Sirketi'nin Sahibi, Siber Operasyon Baskani ve Yonetim Kurulu Baskanisin."""

def format_with_identity(example):
    identity = example.get("system", GLASSES_IDENTITY)
    instruction = example.get("instruction", "")
    output = example.get("output", "")
    return {"text": f"### System\n{identity}\n\n### Instruction\n{instruction}\n\n### Response\n{output}"}

dataset = dataset.map(format_with_identity)
print(f"Identity muhurlendi — {len(dataset)} ornek")

In [ ]:
# 6. EGITIMI BASLAT
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

EPOCHS = 5
BATCH_SIZE = 2
LR = 2e-4
OUTPUT_DIR = "glitch_opus_model"

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    warmup_steps=10,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir=OUTPUT_DIR,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,
)

# Hijyen molasi oncesi:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

trainer.train()
print("EGITIM TAMAMLANDI!")

In [ ]:
# 7. Model kaydet + GGUF
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# GGUF — Ollama'ya yuklemek icin
model.save_pretrained_gguf(
    OUTPUT_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF kaydedildi")

# GGUF dosyasini indir
from google.colab import files
files.download(f"{OUTPUT_DIR}/unsloth.Q4_K_M.gguf")

## Colab'dan Sonra — Lokale Al

1. GGUF dosyasini indir -> `glitch_opus.gguf`
2. `notepad gulmzcetiner\\Modelfile_GLITCH_OPUS` ac, en ust satira ekle:
   ```
   FROM ./glitch_opus.gguf
   ```
3. Modeli olustur:
   ```bash
   ollama create glitch_opus -f gulmzcetiner\\Modelfile_GLITCH_OPUS
   ```
4. Push:
   ```bash
   ollama push glassesglitchstudio/glitch_opus:X_GLITCH_OPUS_V2
   ```
5. GitHub:
   ```bash
   git add glitch_dataset.json glitch_opus.gguf && git commit -m "X_GLITCH_OPUS_V2: QLoRA fine-tune" && git push
   ```